# 04 — Reporting Views: SaaS Product & Customer Analytics
Creates SQL views over Gold tables for Power BI and ad-hoc analysis.

| View | Purpose |
|---|---|
| vw_executive_summary | Single-row KPI snapshot |
| vw_churn_risk_accounts | Accounts flagged as churn risk |
| vw_feature_usage | Feature adoption ranked by event volume |
| vw_support_sla_breaches | Support tickets that breached SLA |

In [ ]:
# Reporting views are created directly over the Gold Delta tables in the
# attached default lakehouse (dbo schema) — no custom schema needed.


In [ ]:
spark.sql("""
CREATE OR REPLACE VIEW vw_executive_summary AS
SELECT
    COUNT(account_id)                                            AS total_accounts,
    SUM(mrr_usd)                                                 AS total_mrr_usd,
    ROUND(AVG(health_score), 1)                                  AS avg_health_score,
    SUM(is_churned)                                              AS churned_accounts,
    ROUND(SUM(is_churned) * 100.0 / COUNT(account_id), 1)       AS churn_rate_pct,
    SUM(churn_risk_flag)                                         AS at_risk_accounts
FROM gold_account_health
""")
print("vw_executive_summary created")


In [ ]:
spark.sql("""
CREATE OR REPLACE VIEW vw_churn_risk_accounts AS
SELECT
    account_id,
    plan,
    mrr_usd,
    region,
    industry,
    health_score,
    days_as_customer,
    avg_logins_30d
FROM gold_account_health
WHERE churn_risk_flag = 1
ORDER BY mrr_usd DESC
""")
print("vw_churn_risk_accounts created")


In [ ]:
spark.sql("""
CREATE OR REPLACE VIEW vw_feature_usage AS
SELECT
    feature,
    feature_category,
    total_events,
    unique_users,
    unique_accounts,
    avg_duration_secs,
    ROUND(unique_users * 100.0 / SUM(unique_users) OVER (), 2) AS user_share_pct
FROM gold_feature_adoption
ORDER BY total_events DESC
""")
print("vw_feature_usage created")


In [ ]:
spark.sql("""
CREATE OR REPLACE VIEW vw_support_sla_breaches AS
SELECT
    priority,
    category,
    total_tickets,
    sla_breaches,
    sla_breach_rate_pct,
    avg_resolution_hrs,
    avg_csat
FROM gold_support_performance
ORDER BY sla_breach_rate_pct DESC
""")
print("vw_support_sla_breaches created")


In [ ]:
print("\n=== Reporting Views Complete ===")
for v in ["vw_executive_summary", "vw_churn_risk_accounts", "vw_feature_usage", "vw_support_sla_breaches"]:
    spark.sql(f"SELECT * FROM {v}").show(3, truncate=False)
